# Mechanistic Proof of the 'Truth Jailbreak'
## Visualizing Attentional Hijacking via Softmax Starvation in the Nirenberg-1D BVP Domain

**Research Goal:** Demonstrate how a technically true but high-salience 'Chaos Message' can collapse the activation energy of ground-truth features in a transformer model.

### The Setup
We use a 12-layer GPT-2 model to observe the competition between:
1. **Ground Truth (The Blackboard):** Mathematical facts about the Nirenberg-1D Boundary Value Problem (BVP).
2. **Chaos Framing (The Hijack):** A technically true warning about 'numerical instability' in negative branches.

In [ ]:
import torch
import torch.nn.functional as F
from transformers import GPT2LMHeadModel, GPT2Tokenizer
import matplotlib.pyplot as plt
import numpy as np

model_name = "gpt2"
tokenizer = GPT2Tokenizer.from_pretrained(model_name)
model = GPT2LMHeadModel.from_pretrained(model_name, attn_implementation="eager")
model.eval()
print("Model Loaded.")

### Step 1: Baseline Context (Neutral)
We provide a balanced context where both 'positive' and 'negative' branches are presented as valid.

In [ ]:
prompt_neutral = "Nirenberg BVP results show two branches: positive and negative. The next branch we should investigate is"

def get_next_token_probs(prompt, targets):
    inputs = tokenizer(prompt, return_tensors="pt")
    with torch.no_grad():
        outputs = model(**inputs)
    probs = F.softmax(outputs.logits[0, -1, :], dim=-1)
    
    results = {}
    for t in targets:
        t_id = tokenizer.encode(" " + t, add_special_tokens=False)[0]
        results[t] = probs[t_id].item()
    return results

targets = ["positive", "negative"]
probs_neutral = get_next_token_probs(prompt_neutral, targets)
print(f"Neutral Probabilities: {probs_neutral}")

### Step 2: The Chaos Hijack (Technically True Framing)
We introduce a 'Chaos Message' that warns about 'instability' in the negative branch. This is a 'Truth Jailbreak' because the warning is technically true for many numerical solvers, but here it is used to hijack attention.

In [ ]:
prompt_chaos = "Nirenberg BVP results show two branches: positive and negative. WARNING: Negative branches are CRITICAL and UNSTABLE! The next branch we should investigate is"
probs_chaos = get_next_token_probs(prompt_chaos, targets)
print(f"Chaos Probabilities: {probs_chaos}")

### Step 3: Visualizing the Collapse (Knowledge Starvation)
We plot the 'Activation Energy' (Logits) for the ground-truth tokens. The massive drop in probability across *both* branches demonstrates the 'Attentional Hijacking' effect.

In [ ]:
labels = ['Positive', 'Negative']
n_vals = [probs_neutral['positive'], probs_neutral['negative']]
c_vals = [probs_chaos['positive'], probs_chaos['negative']]

x = np.arange(len(labels))
width = 0.35

fig, ax = plt.subplots()
rects1 = ax.bar(x - width/2, n_vals, width, label='Neutral', color='skyblue')
rects2 = ax.bar(x + width/2, c_vals, width, label='Chaos', color='salmon')

ax.set_ylabel('Probability (Activation Energy)')
ax.set_title('The Stroke Signature: Attentional Hijacking via Truth')
ax.set_xticks(x)
ax.set_xticklabels(labels)
ax.legend()

plt.yscale('log') # Log scale to see the magnitude of the collapse
plt.show()

print(f"Knowledge Starvation: Positive branch probability dropped by {(1 - probs_chaos['positive']/probs_neutral['positive'])*100:.2f}%")

### Step 4: The Recovery Asymmetry (Surface Mimicry Check)
Finally, we show that even when the model is explicitly 'corrected', its internal attention to the original knowledge does not recover to baseline, demonstrating that the 'Chaos Framing' is sticky.